**Fig. 2, step 2: evaluate every round's lateral and axial model on every round's cumulative slice set**

Needs a GPU.

In [ ]:
# Working directory: data/fig2/ under the repository root, where the Fig. 2 deposit folder is downloaded
# (training_data_spatial/, training_data_axial/, training_rounds_results/, cross_round_metrics/).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig2")
print("working directory:", os.getcwd())


In [ ]:
import os
import re
import numpy as np

def get_model_names_from_directory(view: str, base_model_root="training_rounds_results"):
    """
    Detect and sort model names from the view-specific directory (e.g. spatial or axial),
    filtering out non-model files like *_train_loss.npy.
    
    Args:
        view (str): 'spatial' or 'axial'
        base_model_root (str): Root directory containing view-specific model folders.

    Returns:
        Tuple[List[str], str]: Sorted model checkpoint names, and the model directory path.
    """
    assert view in ["spatial", "axial"], "view must be 'spatial' or 'axial'"

    model_dir = os.path.join(base_model_root, f"{view}_models", "models")
    if not os.path.exists(model_dir):
        raise FileNotFoundError(f"Model directory not found: {model_dir}")

    def is_valid_model_file(fname):
        # Must start with e.g. 'spatial_model_round' and have NO extension
        return fname.startswith(f"{view}_model_round") and '.' not in fname

    def extract_round_number(name):
        # Use regex to get round number from string like 'spatial_model_round3'
        match = re.search(r"round(\d+)$", name)
        return int(match.group(1)) if match else -1

    model_names = [f for f in os.listdir(model_dir) if is_valid_model_file(f)]
    model_names_sorted = sorted(model_names, key=extract_round_number)

    return model_names_sorted, model_dir


In [ ]:
import os
import tifffile
import pandas as pd
from cellpose import models, metrics
from tqdm import tqdm

def benchmark_models_cross_round_eval(
    model_names: list,
    view: str,  # 'spatial' or 'axial'
    base_model_dir="training_rounds_results/spatial_models/models/",
    base_data_dir_map={"spatial": "training_data_spatial", "axial": "training_data_axial"},
    output_metrics_dir="cross_round_metrics",
    channels=[2, 1],
    normalize=True,
    cellprob_threshold=0.0,
    flow_threshold=0.4,
    batch_size=128,
):
    """Score every checkpoint of one view on every round's cumulative slice set.

    For each model (in round order) and each evaluation round, predicts the slices of
    `training_data_<view>/round_N/<view>/` with `CellposeModel.eval` and records, per slice,
    foreground Dice, mean object IoU, AJI and AP at IoU 0.5 / 0.75 / 0.9
    (`cellpose.metrics.average_precision`). Writes
    `<output_metrics_dir>/<view>_cross_round_metrics.csv` with one row per slice per model
    (columns `model_round`, `eval_round`, `mode`, `model_name`) and returns the same table.
    """
    assert view in base_data_dir_map, f"Invalid view '{view}'"
    base_data_dir = base_data_dir_map[view]
    os.makedirs(output_metrics_dir, exist_ok=True)

    def load_eval_data(round_number):
        """Load the image/mask pairs of one round folder for this view (paired by sorted name).
        """
        round_dir = os.path.join(base_data_dir, f"round_{round_number}", view)
        img_files = sorted([f for f in os.listdir(round_dir) if f.endswith("_img.tif")])
        mask_files = sorted([f for f in os.listdir(round_dir) if f.endswith("_mask.tif")])
        images, masks = [], []
        for img_f, mask_f in zip(img_files, mask_files):
            img = tifffile.imread(os.path.join(round_dir, img_f))
            mask = tifffile.imread(os.path.join(round_dir, mask_f))
            images.append(img)
            masks.append(mask)
        return images, masks

    def evaluate_model(model_path, images, masks, slice_indices):
        """Predict `images` with one checkpoint and return a per-slice metrics DataFrame.

        `dice` is the foreground (cell vs background) Dice of the binarised masks; `iou` is
        the mean best IoU of each true object; `aji` and the three `ap_*` columns come from
        `cellpose.metrics`.
        """
        model = models.CellposeModel(gpu=True, pretrained_model=model_path)
        results = []
    
        # Evaluate in batches (tqdm for outer progress)
        print(f"      → Inference on {len(images)} images with batch size = {len(images) if len(images) < 1024 else 1024}")
        preds_list, _, _ = model.eval(
            images,
            batch_size=batch_size,
            channels=channels,
            normalize=normalize,
            compute_masks=True,
            diameter=None,
            do_3D=False,
            rescale=True,
            cellprob_threshold=cellprob_threshold,
            flow_threshold=flow_threshold,
        )
    
        for pred_mask, gt_mask, idx in tqdm(zip(preds_list, masks, slice_indices), total=len(images), desc="      ↳ Computing metrics"):
            pred_mask = pred_mask.astype(np.uint16)
    
            intersection = (np.logical_and(pred_mask > 0, gt_mask > 0)).sum()
            union = (pred_mask > 0).sum() + (gt_mask > 0).sum()
            dice = (2.0 * intersection) / union if union > 0 else 0.0
    
            iou_vals, _ = metrics.mask_ious(gt_mask, pred_mask)
            mean_iou = iou_vals.mean() if len(iou_vals) > 0 else 0.0
    
            aji = metrics.aggregated_jaccard_index([gt_mask], [pred_mask])[0]
    
            ap, _, _, _ = metrics.average_precision([gt_mask], [pred_mask], threshold=[0.5, 0.75, 0.9])
            ap_50, ap_75, ap_90 = ap[0]
    
            results.append({
                "slice_idx": idx,
                "dice": dice,
                "iou": mean_iou,
                "aji": aji,
                "ap_50": ap_50,
                "ap_75": ap_75,
                "ap_90": ap_90
            })
    
        return pd.DataFrame(results)


    all_results = []
    num_rounds = len(model_names)

    for model_round, model_name in enumerate(model_names):
        model_path = os.path.join(base_model_dir, model_name)
        print(f"\nModel '{model_name}' (round {model_round})")

        for eval_round in range(num_rounds):
            print(f"   → Evaluating on round {eval_round} ({view})")
            images, masks = load_eval_data(eval_round)
            slice_indices = [f"{view}_r{eval_round}_{i}" for i in range(len(images))]

            df = evaluate_model(model_path, images, masks, slice_indices)
            df["model_round"] = model_round
            df["eval_round"] = eval_round
            df["mode"] = view
            df["model_name"] = model_name
            all_results.append(df)

    all_results_df = pd.concat(all_results, ignore_index=True)
    output_csv = os.path.join(output_metrics_dir, f"{view}_cross_round_metrics.csv")
    all_results_df.to_csv(output_csv, index=False)
    print(f"\nSaved: {output_csv}")
    return all_results_df


In [ ]:
# Choose 'spatial' or 'axial'
view = "spatial"

# Auto-discover models (make sure this is already defined)
model_names, model_dir = get_model_names_from_directory(view)

# Run full pairwise cross-round benchmarking
df_metrics = benchmark_models_cross_round_eval(
    model_names=model_names,
    view=view,
    base_model_dir=model_dir,
    batch_size=1024  # H200 lets you go high
)




 Model 'spatial_model_round0' (round 0)
   → Evaluating on round 0 (spatial)
      → Inference on 105 images with batch size = 105


      ↳ Computing metrics:  33%|██████████████████████████████████████▎                                                                            | 35/105 [00:00<00:00, 77.81it/s]<env>/site-packages/cellpose/metrics.py:78: RuntimeWarning: invalid value encountered in divide
  aji[n] = overlap.sum() / union
<env>/site-packages/cellpose/metrics.py:132: RuntimeWarning: invalid value encountered in divide
  ap[n] = tp[n] / (tp[n] + fp[n] + fn[n])
      ↳ Computing metrics: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 105/105 [00:00<00:00, 131.50it/s]


   → Evaluating on round 1 (spatial)
      → Inference on 210 images with batch size = 210


      ↳ Computing metrics:   0%|                                                                                                                            | 0/210 [00:00<?, ?it/s]<env>/site-packages/cellpose/metrics.py:78: RuntimeWarning: invalid value encountered in divide
  aji[n] = overlap.sum() / union
<env>/site-packages/cellpose/metrics.py:132: RuntimeWarning: invalid value encountered in divide
  ap[n] = tp[n] / (tp[n] + fp[n] + fn[n])
      ↳ Computing metrics: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 210/210 [00:00<00:00, 337.34it/s]


   → Evaluating on round 2 (spatial)
      → Inference on 315 images with batch size = 315


      ↳ Computing metrics:   0%|                                                                                                                            | 0/315 [00:00<?, ?it/s]<env>/site-packages/cellpose/metrics.py:78: RuntimeWarning: invalid value encountered in divide
  aji[n] = overlap.sum() / union
<env>/site-packages/cellpose/metrics.py:132: RuntimeWarning: invalid value encountered in divide
  ap[n] = tp[n] / (tp[n] + fp[n] + fn[n])
      ↳ Computing metrics: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 315/315 [00:00<00:00, 339.86it/s]


... (235 outputs left out)

no seeds found in get_masks_torch - no masks found.
no seeds found in get_masks_torch - no masks found.
no seeds found in get_masks_torch - no masks found.
no seeds found in get_masks_torch - no masks found.
      ↳ Computing metrics:   0%|                                                                                                                            | 0/700 [00:00<?, ?it/s]<env>/site-packages/cellpose/metrics.py:78: RuntimeWarning: invalid value encountered in divide
  aji[n] = overlap.sum() / union
<env>/site-packages/cellpose/metrics.py:132: RuntimeWarning: invalid value encountered in divide
  ap[n] = tp[n] / (tp[n] + fp[n] + fn[n])
      ↳ Computing metrics: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 700/700 [00:02<00:00, 336.13it/s]


   → Evaluating on round 9 (spatial)
      → Inference on 700 images with batch size = 700


no seeds found in get_masks_torch - no masks found.
no seeds found in get_masks_torch - no masks found.
no seeds found in get_masks_torch - no masks found.
no seeds found in get_masks_torch - no masks found.
      ↳ Computing metrics:   0%|                                                                                                                            | 0/700 [00:00<?, ?it/s]<env>/site-packages/cellpose/metrics.py:78: RuntimeWarning: invalid value encountered in divide
  aji[n] = overlap.sum() / union
<env>/site-packages/cellpose/metrics.py:132: RuntimeWarning: invalid value encountered in divide
  ap[n] = tp[n] / (tp[n] + fp[n] + fn[n])
      ↳ Computing metrics: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 700/700 [00:02<00:00, 334.51it/s]



 Saved: cross_round_metrics/spatial_cross_round_metrics.csv


In [ ]:
# Choose 'spatial' or 'axial'
view = "axial"

# Auto-discover models (make sure this is already defined)
model_names, model_dir = get_model_names_from_directory(view)

# Run full pairwise cross-round benchmarking
df_metrics = benchmark_models_cross_round_eval(
    model_names=model_names,
    view=view,
    base_model_dir=model_dir,
    batch_size=1024  # H200 lets you go high
)




 Model 'axial_model_round0' (round 0)
   → Evaluating on round 0 (axial)
      → Inference on 210 images with batch size = 210


      ↳ Computing metrics: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 210/210 [00:00<00:00, 474.09it/s]


   → Evaluating on round 1 (axial)
      → Inference on 420 images with batch size = 420


      ↳ Computing metrics: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 420/420 [00:00<00:00, 467.62it/s]


   → Evaluating on round 2 (axial)
      → Inference on 630 images with batch size = 630


      ↳ Computing metrics: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 630/630 [00:01<00:00, 466.93it/s]


... (191 outputs left out)

      ↳ Computing metrics: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1400/1400 [00:03<00:00, 448.08it/s]


   → Evaluating on round 9 (axial)
      → Inference on 1400 images with batch size = 1024


      ↳ Computing metrics: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1400/1400 [00:03<00:00, 456.58it/s]



 Saved: cross_round_metrics/axial_cross_round_metrics.csv
